MultiModal RAG

In [78]:
import fitz #Pymupdf
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
import torch
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import pandas as pd

In [79]:
#load the CLIP model
from dotenv import load_dotenv
load_dotenv()

#set up environment
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
#intilize the CLIP model for unified embedding
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 22314.60it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [80]:
model=init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026B9009E8A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026AD184F320>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [81]:
#Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data,str): #if image path is given(URL)
        image=Image.open(image_data).convert("RGB")
    else: #if image_data is image
        image=image_data
    
    inputs=clip_processor(images=image,return_tensors="pt")

    with torch.no_grad():
        #takes embeddings 
        features=clip_model.get_image_features(**inputs)
        features=features.pooler_output
        #Normalize embeddings to unit vector
        features=features/features.norm(dim=-1,keepdim=True)
    return features.squeeze().numpy()

def embed_text(text):
    """Embed text using CLIP"""
    inputs=clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77   #CLIP's max token length
    )
    with torch.no_grad():
        features=clip_model.get_text_features(**inputs)
        features=features.pooler_output
        #print(features)
        #Normalize embeddings
        features=features/features.norm(dim=-1,keepdim=True)
        return features.squeeze().numpy()
        



In [82]:
#process pdf
pdf_path="attention-is-all-you-need.pdf"
#pdf_path="attention-is-all-you-need.pdf"
docs=fitz.open(pdf_path)

#storage for all docs and embeddings
all_docs=[]
all_embeddings=[]
image_data_store={} #store actual image data for llm

#text splitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [83]:
docs

Document('attention-is-all-you-need.pdf')

In [84]:
for i,page in enumerate(docs):
    #process text
    text=page.get_text()
    if text.strip():
        #create temporary document for splitting
        temp_doc=Document(page_content=text, metadata={"page":i,"type":"text"})
        text_chunks=splitter.split_documents([temp_doc])

        #embed each chuk using CLIP
        for chunk in text_chunks:
            embedding=embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)
    
    try:
        tables=page.find_tables()

        for table_idx,table in enumerate(tables.tables):
            table_data=table.extract()

            if table_data:
                df=pd.DataFrame(table_data[1:],columns=table_data[0])
                table_txt=df.to_markdown(index=False)

                print("=" * 50)
                print(table_txt)
                print("=" * 50)


                table_doc=Document(
                    page_content=table_txt,
                        metadata={
                        "page":i,
                        "type":"table",
                        "table_id":table_idx
                        }
                        )

                embedding=embed_text(table_txt)

                all_embeddings.append(embedding)
                all_docs.append(table_doc)
                
    except Exception as e:
        print(f"Error processing table {table_idx} on page {i}: {e}")


    #process image

    #1. convert PDF image to PIL format
    #2. store as base64 for LLM
    #3. create CLIP embeddingfor retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref=img[0]
            base_img=docs.extract_image(xref)
            image_bytes=base_img["image"]

            #convert to PIL image
            pil_img=Image.open(io.BytesIO(image_bytes)).convert("RGB")

            #create unique identifier
            image_id=f"page_{i}_img_{img_index}"

            #store image as base64 for later use with LLM
            buffered=io.BytesIO()
            pil_img.save(buffered,format="PNG")
            img_base64=base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id]=img_base64

            #embed image using CLIP
            embedding=embed_image(pil_img)
            all_embeddings.append(embedding)

            #create document for image
            img_doc=Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page":i, "type":"image","image_id":image_id}
            )
            all_docs.append(img_doc)
        
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

docs.close()







|      | train                                     | PPL BLEU params    |
|      | N d d h d d P ϵ                           | (dev) (dev) ×106   |
|      | model ff k v drop ls steps                |                    |
|:-----|:------------------------------------------|:-------------------|
| base | 6 512 2048 8 64 64 0.1 0.1 100K           | 4.92 25.8 65       |
| (A)  | 1 512 512                                 | 5.29 24.9          |
|      | 4 128 128                                 | 5.00 25.5          |
|      | 16 32 32                                  | 4.91 25.8          |
|      | 32 16 16                                  | 5.01 25.4          |
| (B)  | 16                                        | 5.16 25.1 58       |
|      | 32                                        | 5.01 25.4 60       |
| (C)  | 2                                         | 6.11 23.7 36       |
|      | 4                                         | 5.19 25.3 50       |
|      | 8                            

In [85]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu'),
 Document(metadata={'page': 0, 'type': 'text'}, page_content='Google Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The

In [86]:
#create unified FAISS vector store with CLIP embeddings
embedding_array=np.array(all_embeddings)
embedding_array

array([[ 0.03500306,  0.00810928, -0.03526159, ...,  0.02110534,
         0.0016752 , -0.03087727],
       [ 0.00678681, -0.00609663, -0.0433337 , ..., -0.03352812,
        -0.02618885, -0.01386962],
       [ 0.00037515, -0.02122331, -0.00457746, ..., -0.01093403,
         0.0380313 , -0.02663142],
       ...,
       [-0.01928525,  0.00096109,  0.00146911, ..., -0.03778002,
         0.02002626,  0.00973881],
       [-0.01122869, -0.02954053,  0.02860653, ..., -0.03015947,
         0.02209018,  0.00036677],
       [ 0.03821684, -0.00743307,  0.00681705, ..., -0.02220306,
         0.00526421, -0.00230353]], shape=(112, 512), dtype=float32)

In [87]:
(all_docs,all_embeddings)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu'),
  Document(metadata={'page': 0, 'type': 'text'}, page_content='Google Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. T

In [88]:


#create custom FAISS index since we have precomputed embeddings
vector_store=FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embedding_array)],
    embedding=None,  #we using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [89]:
def retrive_model(query,k=5):
    """Unified retrieval using CLIP embedding for both text and images"""
    #embed query using CLIP
    query_embedding=embed_text(query)

    #search in unified vector store
    results=vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

In [90]:
def create_multimodal_message(query,retrieved_docs):
    """create a message with both text and images for LLM"""
    content=[]

    #add the query
    content.append({
        "type" : "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    #separate text and image documents
    text_docs=[doc for doc in retrieved_docs if doc.metadata.get("type")=="text"]
    table_docs=[doc for doc in retrieved_docs if doc.metadata.get("type")=="table"]
    image_docs=[doc for doc in retrieved_docs if doc.metadata.get("type")=="image"]

    #add text content
    if text_docs:
        text_context="\n\n".join([
            f"[Page {doc.metadata['page']}]; {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type":"text",
            "text":f"text excepts:\n{text_context}\n"
        })
    
    if table_docs:
        table_context="\n\n".join([
            f"[Table on page {doc.metadata['page']}]; {doc.page_content}"
            for doc in table_docs
        ])
        content.append({
                "type":"text",
                "text":f"Tables:\n{table_context}\n"
        })

    #Add images
    for doc in image_docs:
        image_id=doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type":"text",
                "text":f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type":"image_url",
                "image_url":{
                    "url":f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    #Add instruction
    content.append({
        "type":"text",
        "text":"\n\nPlease answer the question based on the provided text and images."
    })
    return HumanMessage(content=content)

In [91]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal rag."""

    #Retrive relevant documents
    context_docs=retrive_model(query,k=5)

    #create multimodal message
    message=create_multimodal_message(query,context_docs)

    #get response from LLM
    response=model.invoke([message])

    #print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type=doc.metadata.get("type","unknown")
        page=doc.metadata.get("page","?")
        if doc_type=="text":
            preview=doc.page_content[:100] + "..." if len(doc.page_content)>100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        elif doc_type=="table":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f" - Table from page {page}: {preview}")
        else:
            print(f"   - Image from page {page}")
    print("\n")

    return response.content

In [92]:
if __name__ == "__main__":
    # queries=[
    #     "What does the chart on page 0 show about revenue trends?",
    #     "summarize the main findings from the document",
    #     "What visual element are present in the document?"
    # ]
    queries=[
       "How many attention heads does the Transformer use, and what is the dimension of each head? "
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer=multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("*" * 70)


Query: How many attention heads does the Transformer use, and what is the dimension of each head? 
--------------------------------------------------

Retrieved 5 documents:
  - Text from page 1: To the best of our knowledge, however, the Transformer is the first transduction model relying
entir...
 - Table from page 12: | It   | is   | in   | is   | rit   | at   | a   | ty   | of   | n   | ts   | e   | d   | w   | s   ...
  - Text from page 8: (section 5.4), learning rates and beam size on the Section 22 development set, all other parameters
...
  - Text from page 8: base
6
512
2048
8
64
64
0.1
0.1
100K
4.92
25.8
65
(A)
1
512
512
5.29
24.9
4
128
128
5.00
25.5
16
32
...
 - Table from page 12: | nan   |    | nan   |    | nan   |    | nan   |    | nan   |    | nan   |     | nan   |    | nan   ...


Answer: The Transformer model architecture is described on page 3 and the table on page 12 provides the details of the model.

The base model of the Transformer uses 8 attention heads, and th